<a href="https://colab.research.google.com/github/e23070-Dulara/Statistical-Learning-e23070/blob/main/Copy_of_GPR_LR_assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Gaussian Process Regression

Consider the following [data set](https://www.kaggle.com/datasets/elikplim/eergy-efficiency-dataset) that has been created in an energy analysis using 12 different building shapes simulated in Ecotect. The buildings differ with respect to the glazing area, the glazing area distribution, and the orientation, amongst other parameters. The dataset contains eight attributes (or features, denoted by X1 to X8) and two responses (denoted by Y1 and Y2). Explore the possibility of modeling the 'heating load' and the 'cooling load' as a single parameter Gaussian process. Discuss your conclusions.

In [1]:
import kagglehub

# Download latest version
kagglepath="elikplim/eergy-efficiency-dataset"
path = kagglehub.dataset_download(kagglepath)

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'eergy-efficiency-dataset' dataset.
Path to dataset files: /kaggle/input/eergy-efficiency-dataset


In [2]:
import os
import pandas as pd
print(f"Listing contents of: {path}")
!ls {path}
df2=pd.read_csv(path+"/ENB2012_data.csv")

Listing contents of: /kaggle/input/eergy-efficiency-dataset
ENB2012_data.csv


NameError: name 'pd' is not defined

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import gpytorch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import kagglehub

# 1. Download and Load Dataset
path = kagglehub.dataset_download("elikplim/eergy-efficiency-dataset")
# Find the exact Excel/CSV file path in the downloaded folder
file_name = [f for f in os.listdir(path) if f.endswith('.xlsx') or f.endswith('.csv')][0]
full_path = os.path.join(path, file_name)

# Read dataset (Assuming it's an Excel file based on UCI Energy Efficiency format)
df = pd.read_excel(full_path) if full_path.endswith('.xlsx') else pd.read_csv(full_path)
df.dropna(inplace=True)

# Rename columns based on UCI repository convention
df.columns = ['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8', 'Y1', 'Y2']

# 2. Data Preprocessing
X = df[['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8']].values
Y = df[['Y1', 'Y2']].values

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

scaler_X = StandardScaler()
X_train_scaled = scaler_X.fit_transform(X_train)
X_test_scaled = scaler_X.transform(X_test)

# Convert to PyTorch Tensors
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
Y_train_tensor = torch.tensor(Y_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
Y_test_tensor = torch.tensor(Y_test, dtype=torch.float32)

# 3. Define Multitask / Multi-Output GP Model (ICM Framework)
class MultitaskGPModel(gpytorch.models.ExactGP):
    def __init__(self, train_x, train_y, likelihood):
        super(MultitaskGPModel, self).__init__(train_x, train_y, likelihood)
        self.mean_module = gpytorch.means.MultitaskMean(
            gpytorch.means.ConstantMean(), num_tasks=2
        )
        self.covar_module = gpytorch.kernels.MultitaskKernel(
            gpytorch.kernels.RBFKernel(), num_tasks=2, rank=1
        )

    def forward(self, x):
        mean_x = self.mean_module(x)
        covar_x = self.covar_module(x)
        return gpytorch.distributions.MultitaskMultivariateNormal(mean_x, covar_x)

likelihood = gpytorch.likelihoods.MultitaskGaussianLikelihood(num_tasks=2)
model = MultitaskGPModel(X_train_tensor, Y_train_tensor, likelihood)

# 4. Training the Model
model.train()
likelihood.train()

optimizer = torch.optim.Adam(model.parameters(), lr=0.1)
mll = gpytorch.mlls.ExactMarginalLogLikelihood(likelihood, model)

training_iterations = 150
print("--- Starting Training ---")
for i in range(training_iterations):
    optimizer.zero_grad()
    output = model(X_train_tensor)
    loss = -mll(output, Y_train_tensor)
    loss.backward()
    if (i + 1) % 25 == 0:
        print(f"Iter {i + 1}/{training_iterations} - Loss: {loss.item():.4f}")
    optimizer.step()

# 5. Evaluation & Predictions
model.eval()
likelihood.eval()

with torch.no_grad(), gpytorch.settings.fast_pred_var():
    predictions = likelihood(model(X_test_tensor))
    mean = predictions.mean
    lower, upper = predictions.confidence_region()

# Calculate Performance Metrics (MSE)
mse_heating = torch.mean((mean[:, 0] - Y_test_tensor[:, 0]) ** 2).item()
mse_cooling = torch.mean((mean[:, 1] - Y_test_tensor[:, 1]) ** 2).item()

print("\n--- Model Evaluation ---")
print(f"Heating Load (Y1) Test MSE: {mse_heating:.4f}")
print(f"Cooling Load (Y2) Test MSE: {mse_cooling:.4f}")

# Linear Regression

Consider the following [data set](https://www.kaggle.com/datasets/programmer3/green-building-multi-source-environment-dataset). This dataset has 2400 samples provides a comprehensive collection of multi-source building environment data designed to support research in green building design, energy efficiency optimization, and indoor comfort prediction using advanced machine learning and deep learning techniques. Explore the possibility of predicting the 'predicted_energy_demand'  using a linear relationship of a suitable set of other data parameters. Justify your choice of parameters and discuss the results.

In [3]:
import kagglehub

# Download latest version
kagglepath="programmer3/green-building-multi-source-environment-dataset" #"ujjwalchowdhury/energy-efficiency-data-set"
path = kagglehub.dataset_download(kagglepath)

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'green-building-multi-source-environment-dataset' dataset.
Path to dataset files: /kaggle/input/green-building-multi-source-environment-dataset


In [4]:
import os
print(f"Listing contents of: {path}")
!ls {path}
df2=pd.read_csv(path+"/green_building_dataset.csv")
inspector.df=df2

Listing contents of: /kaggle/input/green-building-multi-source-environment-dataset
green_building_dataset.csv


NameError: name 'pd' is not defined

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import kagglehub

# 1. Fetch and load the Kaggle Dataset
kagglepath = "programmer3/green-building-multi-source-environment-dataset"
path = kagglehub.dataset_download(kagglepath)
print("Path to dataset files:", path)

# Dynamically locate the CSV target file
csv_files = glob.glob(os.path.join(path, "**/*.csv"), recursive=True)
if not csv_files:
    raise FileNotFoundError("No CSV file discovered in the downloaded dataset directory.")
full_path = csv_files[0]
print(f"Loading data asset from: {full_path}")

df = pd.read_csv(full_path)

# 2. Inspect target variable mapping
# Adjust name slightly if casing variants exist in runtime environment
target_col = 'predicted_energy_demand'
if target_col not in df.columns:
    # Fail-safe fallbacks if column naming contains alternate capitalization strings
    cols_lower = {c.lower(): c for c in df.columns}
    if target_col in cols_lower:
        target_col = cols_lower[target_col]
    else:
        # Dynamic fallback if target column differs slightly
        energy_cols = [c for c in df.columns if 'energy' in c.lower() or 'demand' in c.lower()]
        if energy_cols:
            target_col = energy_cols[0]
        else:
            raise KeyError(f"Could not automatically detect '{target_col}' among data paths.")

print(f"Target column successfully resolved to: '{target_col}'")

# 3. Data Cleaning and Feature Generation
# Drop identifiers that provide zero physical predictive weight
cols_to_drop = [c for c in ['Record_ID', 'Timestamp', 'Building_ID'] if c in df.columns]
df = df.drop(columns=cols_to_drop)

# Separate Target from Features
X = df.drop(columns=[target_col])
y = df[target_col]

# Handle Categorical Features via One-Hot Encoding
X = pd.get_dummies(X, drop_first=True)

# Align Data Alignment Arrays
X = X.apply(pd.to_numeric, errors='coerce')
X.fillna(X.mean(), inplace=True)
y = pd.to_numeric(y, errors='coerce').fillna(y.mean())

# 4. Train-Test Splitting and Feature Scaling
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 5. Execute Linear Regression Model
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)

# 6. Evaluation and Diagnostics
y_pred = lr_model.predict(X_test_scaled)

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("\n=========================================")
print("     LINEAR REGRESSION MODEL RESULTS     ")
print("=========================================")
print(f"Mean Absolute Error (MAE)   : {mae:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")
print(f"Coefficient of Determination (R²): {r2:.4f}")
print("=========================================\n")

# Display sorted feature weight magnitudes
coefficients = pd.DataFrame({
    'Feature': X.columns,
    'Standardized Coefficient': lr_model.coef_
}).sort_values(by='Standardized Coefficient', key=abs, ascending=False)

print("Feature Weights (Sorted by Magnitude Impact):")
print(coefficients.to_string(index=False))